![hslu_logo.png](./img/hslu_logo.png)


<hr style="border:1px solid black">

<h1 style="text-align:center;font-size:50px"><b>AAI - FS25</b></h1>
<p style="text-align:center;font-size:40px">Week 07</p>

---
# CNN Quantization & Lab
---
---
# Table of contents for week 07
1. [Summary EAS and Excercise 6.3](#summary)
   1. [# Weights CIFAR DNA 0](#summary_1)
   2. [Training on Graphics Card ](#summary_2)
   3. [Challenge 1/2 Winners](#summary_3)
2. [Introduction NVIDIA Jetson Orin Nano](#orin)
   1. [Setup Target HW & Toolchain](#orin_1)
   2. [Ampere 100: Architecture](#orin_2)
   3. [Ampere 100: Data Types](#orin_3)
3. [Lab 1: CIFAR Inference from File](#single)
     1. [Model & Test Data Export](#file_1)
     2. [Model Conversion \& Quantization](#file_2)
     3. [Conversion \& Quantization Validation](#file_3)
4. [Lab 2: CIFAR Inference from USB-Camera](#stream)
     1. [Model Export](#stream_1)
     2. [Model Conversion \& Quantization](#stream_2)
     3. [CIFAR Inference in Streaming Mode](#stream_3)



# Summary EAS and Excercise 6.3 <a name="summary"></a>

## # Weights CIFAR DNA 0 <a name="summary_1"></a>

- Train. parameters calculated according to our formular in Excercise 6.1: 550'570
- Train. parameters reported by TensorFlow: 551'466 (896 less)
- Reason: 
  - Formula only takes into account weights from CONV and FC layers
  - TensorFlow also includes train. parameters from other layers, in this case, 896 train. parameters ($\gamma$ and $\beta$) from all batch mormalization layers 
  - Additionally, 896 non-trainable parameters $\mu$ and $\sigma$ of these batch normalization layers reported by TensoFlow, these are data-dependent but not trainable

## Training on Graphics Card <a name="summary_2"></a>

- Sspeed-up (around 10x) can be achieved by installiing the following package, [see here:](https://learn.microsoft.com/en-us/windows/ai/directml/gpu-tensorflow-plugin):

      pip install tensorflow-directml-plugin

- This will downgrade the TesorFlow environment from version 2.13 to 2.10. 
- Result: ONNX export will not work anymore, all other EAS-scripts continue to work
- Workaround: Setup two venv enviroments, with and without tensorflow-directml-plugin


## Challenge 1/2 Winners <a name="summary_3"></a>

- Best networks WaJ
  - Challenge 1: 9'575'856 MACs ( 79.29 %)
  - Challenge 2: 82.93 % (38'054'144 Macs)

# Introduction NVIDIA Jetson Orin Nano <a name="orin"></a>

## Setup Target HW & Toolchain <a name="orin_1"></a>

The embedded target computer is running a Linux OS (Ubuntu) and has the following static IP address:

     192.168.1.100 

A user has been setup as with the following credentials:

     user: aai 
     pass: hslu 

1. To connect to the target configure static IP for Ethernet-Adapter XX on the host notebook:
    - IP address:  192.168.1.101
    - Subnet Mask: 255.255.225.0
2. For transferring files from the host to the target we use *Secure Copy (SSC)*. For this you can <ins>either</ins>:
    - install WinSCP with a explorer-like GUI, [click for download](https://winscp.net/download/WinSCP-6.5-Setup.exe/download), <ins>or</ins>
    - work on the Windows command line using the following syntax to copy a file from the current working directory on the host into the home directory of user aai on the target:

          scp *file_name* aai@192.168.1.100:/home/aai

3. For editing files on the target you can <ins>either</ins>:
    - use Visual Studio Code and open an SSH connection, <ins>or</ins>
    - work on the command line by 
      - Connect via *Secure Shell (SSH)* by typing on the windows command line:
  
            ssh aai@192.168.1.100 

      - and then use a Linux command-line editor, e.g. the [vi Editor](https://www.thomas-krenn.com/en/wiki/Vi_editor_tips_and_tricks) by typing:
  
            vi *file_name*

4. Transfer the neccessary Python scripts for SW07 to the target:
    - On the host command line run:

          scp aai_sw07.zip aai@192.168.1.100:/home/aai

    - On the target command line run:
  
          unzip aai_sw07.zip

    - This creates the following folder structure on the target:

          /home/aai/aai_sw07/
             00_model_conversion
             01_single_inference
             02_streaming_inference


## Ampere 100: Architecture <a name="orin_2"></a>

- NVIDIA Jetson familiy is a series of embedded computing platforms designed for AI and machine learning applications.  
  - 6-core Arm® Cortex-A78 (64-bit CPU)
  - GPU with Ampere architecture

<img src="./img/jetson_family.png" alt="Jetson" width="650px" />  

- See [NVIDIA page](https://www.nvidia.com/en-us/data-center/ampere-architecture/) and [Developer Blog](https://developer.nvidia.com/blog/nvidia-ampere-architecture-in-depth/?ncid=no-ncid) and [White Paper](https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/nvidia-ampere-architecture-whitepaper.pdf) on Ampere 100 Architecture
- Streaming Modules (SM) contain 4 Tensor Cores. Note the # of Registers per Core!

<img src="./img/ga100_sm.png" alt="SM" width="450px" />  


## Ampere 100: Data Types <a name="orin_3"></a>

- see white paper ...


# Lab 1: CIFAR Inference from File <a name="file"></a>

In this lab, you will export your best models from Excercise 6.3 in the generic [ONNX format](https://onnx.ai/), such that it can be quantized and compiled for the target plattform and the executed there.  

## Model & Test Data Export <a name="file_1"></a>

In this step we continue to work on Excercise 6.3 in the AAI venv in folder SW06.

1. Replace *export_model.py* with the new version from ILIAS/SW07.
2. Use the script to export the 2 to 3 best of your networks from Excercise 6.3 with BATCH_SIZE = 10, e.g.:
       
        python export_model.py best_model_dna_17 10
        python export_model.py best_model_dna_21 10

3. The script will do the following:
   1. export the given model in ONNX format with the selected batch size (best_model_dna_XY.onnx).
   2. export 2 files with numpy arrays which we will use later on the target to verify correctness of our exported and compiled models: 
      - model input data from the testset (best_model_dna_XY_input.npy)
      - model output data (predictions) from this input data  (best_model_dna_XY_output.npy)
   3. convert the model and the input data from **channel-last NHWC** ordering  (used in TesnorFlow) to **channel-first NCHW** ordering (used in target environment)
   4. verify the reordering by showing the RED color-channel of the original and the two-times re-verted channel-first version of the last image in the seleted batch.
4. Verify that the above procedure for testing the correct channel-first/last conversion is effective, by deliberately showing different color channels in the original and two-times reverted image. 
5. To see the model architecture before and after export you can drag-and-drop the original .h5 file and the converted .onnx file to [NETRON App](https://netron.app/) for comparison.

## Model Conversion \& Quantization <a name="file_2"></a>

1. Transfer the exported files to the following directories on the target

     - .onnx --> /00_model_conversion/models
     - .npy --> /01_single_inference/images
  
2. To quantize and compile all models stored under */00_model_conversion/models* change directory to *00_models_conversion* and run

       python main.py

3. Under */00_model_conversion/engines* this will create for each model two GPU runtime-engines with FP16 and FP32 quantization of weights, respectively:
      - *_fp_16.engine
      - *_fp32.engine

## Conversion \& Quantization Validation <a name="file_3"></a>

To validate the conversion and quantization process of the models just performed, we are going to compare 

- the predictions performed with the ONNX model on the target CPU, and 
- the predictions performed with the fp16/32-engines on the target GPU

with the predictions of the corresponding Keras model form Excercise 6.3 (expected output).

1. Copy the desired (or all) generated engine-files into the working directory for "single inference from file":

       cp 00_model_conversion/engines/* 01_single_inference/engines/

2.  Copy the desired (or all) original ONNX models into the working directory for "single inference from file":

       cp 00_model_conversion/models/* 01_single_inference/models/

3. To perform inference from file change directory to *01_single_inference* and run

       python main.py

4. Compare and discuss the various inferrence errors dispalyed.
5. Change the output (expected result) file used and rerun the inference process. Observe and discuss the changes in the reported inference errors.

**Notes**
- The validation performed here is only concerned with conversion and qunatization of the original Keras model for the target environment. It does **not** provide information on the accuracy performance of the models!
- The reported inference error is the maximum over all *BATCH_SIZE* $\times$ *NUM_CLASSES* output values of a particular model.
- The input (image) and output (expected response) file used for an inference run can be selcetd on lines 12 and 13 of file *01_single_inference/main.py*

      image_path = Path("images/best_model_dna_0_input.npy")
      expected_result_path = Path("images/best_model_dna_0_output.npy")

- It is sufficient to store only one input file under */01_single_inference/images/*, since all of them are identical.

# Lab 2: CIFAR Inference from USB-Camera <a name="file"></a>

In this Lab we are going to run selected models on the GPU in streaming mode, performing predictions on images captured with a USB-camera.

## Model Export <a name="stream_1"></a>

1. Use *export_model.py* to export the 2 to 3 best of your networks from Excercise 6.3 with **BATCH_SIZE = 1**, e.g.:
        python export_model.py best_model_dna_17 1
        python export_model.py best_model_dna_21 1

## Model Conversion \& Quantization <a name="stream_2"></a>

1. Transfer the exported (and maybe renamed) files to the following directories on the target

    - .onnx --> /00_model_conversion/models
  
2. To quantize and compile all models stored under */00_model_conversion/models* change directory to *00_models_conversion* and run

       python main.py

## CIFAR Inference in Streaming Mode <a name="stream_3"></a>

1. Copy **one** generated engine-file with BATCH_SIZE = 1  into the working directory for "streaming inference from USB-Camera", e.g.:

       cp 00_model_conversion/engines/best_model_dna_0_BS1_pyapi_fp_32.engine 02_streaming_inference/engines/

2. To perform streaming inference from the USB-Camera change directory to *02_streaming_inference* and run

       python main.py

3. Open a Web Browser on the host and connect to the target on port 8080:
   
       192.168.1.100:8080

4. Discuss the *processing time* and *Frames per second* metrics displayed. 

5. Experiment with your models by pointing the camera to some of the provided images, e.g.:

<img src="./img/automobile.png" alt="auto" width="450px" />
